# 📝 Resume Generator with Google Drive Integration

<div style="background-color: #f0f7fb; padding: 20px; border-radius: 10px; margin-bottom: 20px; border-left: 5px solid #3498db;">
    <h2 style="margin-top: 0; color: #3498db;">Overview</h2>
    <p>This interactive notebook streamlines your resume updates by:</p>
    <ol>
        <li>Downloading keywords from a Google Sheet</li>
        <li>Processing keywords to find the most relevant ones</li>
        <li>Updating your resume with these keywords</li>
        <li>Generating a professional PDF resume</li>
        <li>Uploading the PDF to Google Drive</li>
    </ol>
    <p><strong>One click to maintain an optimized, keyword-focused resume!</strong></p>
</div>

## 🔧 Configuration

Adjust the settings below to customize your resume generation process:

<div style="background-color: #e8f4f8; padding: 10px; border-radius: 5px; margin-bottom: 15px; border-left: 5px solid #4dabf7;">
    <p><strong>💡 TIP:</strong> Scroll down to <a href="#resume-structure">Resume Structure</a> section to configure your resume section order!</p>
</div>

In [ ]:
# Configuration Settings
config = {
    # Google Drive Integration
    "use_google_drive": False,                          # Set to False to skip Google Drive operations
    "sheet_id": "1C20hzxMQzT310HSe9DhH8XigArW5VcjAGgQfGafF4o4", # Google Sheet ID
    "folder_id": "1s6n9eS9d2NNy9lyXgoYdPGOwkyGGujnw",  # Google Drive folder ID for uploads
    
    # File Paths
    "resume_template": "templates/resume.md",          # Resume template path
    "output_filename": "resume",                       # Base name for output files (without extension)
    
    # Resume Structure
    "structure_config": "templates/resume_config.yaml", # YAML configuration for resume section ordering
    
    # Keyword Processing
    "top_keywords": 100,                                # Number of top keywords to include
    
    # PDF Settings
    "generate_pdf": True,                              # Try to generate PDF with pandoc
    
    # Display Settings
    "verbose_output": True,                            # Show detailed processing logs
    "show_keyword_count": 15                           # Number of keywords to display in results
}

# Display the configuration
print("Configuration loaded successfully.")

## 🚀 Generate Your Resume

Run this cell to execute the entire resume generation process using the configuration above. The process includes:

1. Downloading keywords data from Google Drive (if enabled)
2. Processing keywords to identify the most relevant terms
3. **Applying your customized resume structure** from the configuration
4. Enhancing your resume with identified keywords 
5. Generating a professional PDF
6. Uploading the PDF to Google Drive (if enabled)

For a fully tailored resume, first use the [Resume Structure](#resume-structure) section below to customize your section ordering before running this cell.

In [ ]:
import os
import sys
import time
import datetime
import pandas as pd
from IPython.display import HTML, display, Markdown, FileLink
import ipywidgets as widgets
import yaml
import importlib

# Setup paths
notebook_dir = os.path.abspath('')
base_dir = notebook_dir
scripts_dir = os.path.join(base_dir, 'scripts')

# Add scripts directory to path
sys.path.append(base_dir)
sys.path.append(scripts_dir)

# Import custom modules
import scripts.keywords_processor as keywords_processor
import scripts.update_resume as update_resume

# Force reload resume_generator module to get the latest version
if 'scripts.resume_generator' in sys.modules:
    importlib.reload(sys.modules['scripts.resume_generator'])
import scripts.resume_generator as resume_generator

# Check for Google Drive integration
if config["use_google_drive"]:
    try:
        import scripts.google_drive_handler as google_drive
        google_drive_available = True
    except ImportError:
        print("⚠️ Google Drive integration unavailable. Continuing without it.")
        google_drive_available = False
else:
    google_drive_available = False

# Configure file paths based on settings
input_keywords_file = os.path.join(base_dir, "data/input/lead_gen.csv")
processed_keywords_file = os.path.join(base_dir, "data/output/processed_keywords.csv")
input_resume_file = os.path.join(base_dir, config["resume_template"])
output_resume_file = os.path.join(base_dir, f"data/output/{config['output_filename']}.md")
output_pdf_file = os.path.join(base_dir, f"data/output/{config['output_filename']}.pdf")
config_file = os.path.join(base_dir, "templates/resume_config.yaml")

# Create directories if they don't exist
os.makedirs(os.path.dirname(input_keywords_file), exist_ok=True)
os.makedirs(os.path.dirname(output_resume_file), exist_ok=True)

# Styled logging function
def log(message, level="INFO", display_only=False):
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    if level == "INFO":
        color = "#3498db"  # Blue
        icon = "ℹ️"
    elif level == "SUCCESS":
        color = "#2ecc71"  # Green
        icon = "✅"
    elif level == "WARNING":
        color = "#f39c12"  # Orange
        icon = "⚠️"
    elif level == "ERROR":
        color = "#e74c3c"  # Red
        icon = "❌"
    else:
        color = "#7f8c8d"  # Gray
        icon = "🔹"
    
    html = f"<div style='margin-bottom: 8px;'><span style='color: {color}; font-weight: bold;'>{icon} {level}</span> <span style='color: #7f8c8d; font-size: 0.9em;'>[{timestamp}]</span> {message}</div>"
    display(HTML(html))
    
    if not display_only and config["verbose_output"]:
        print(f"[{timestamp}] {level}: {message}")

# Show a section header
def show_section(title, description=None):
    display(HTML(f"""
    <div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; margin: 20px 0 10px 0; border-left: 5px solid #6c757d;">
        <h3 style="margin: 0; color: #343a40;">{title}</h3>
        {f'<p style="margin: 5px 0 0 0; color: #6c757d;">{description}</p>' if description else ''}
    </div>
    """))

# Show a summary box 
def show_summary_box(title, items):
    html = f"""
    <div style="background-color: #f8f9fa; padding: 15px; border-radius: 5px; margin: 10px 0; border: 1px solid #dee2e6;">
        <h4 style="margin-top: 0; color: #343a40;">{title}</h4>
        <ul style="margin-bottom: 0; padding-left: 20px;">
    """
    
    for item in items:
        html += f"<li>{item}</li>"
    
    html += """
        </ul>
    </div>
    """
    
    display(HTML(html))

# Show keywords in a nice table
def show_keywords_table(df, top_n=10, title="Top Keywords"):
    # Create a copy with limited rows
    display_df = df.head(top_n).copy()
    
    # Style the dataframe
    styled_df = display_df.style.set_caption(title)\
        .set_table_styles([
            {'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '1.1em'), ('color', '#343a40')]},
            {'selector': 'th', 'props': [('background-color', '#f8f9fa'), ('color', '#343a40')]},
            {'selector': 'td', 'props': [('padding', '5px 10px')]}
        ])\
        .set_properties(**{'text-align': 'left'})
    
    display(styled_df)

# Display file link with nice formatting
def show_file_link(file_path, description):
    if os.path.exists(file_path):
        file_name = os.path.basename(file_path)
        file_size = os.path.getsize(file_path) / 1024  # KB
        size_str = f"{file_size:.1f} KB" if file_size < 1024 else f"{file_size/1024:.1f} MB"
        
        display(HTML(f"""
        <div style="margin: 10px 0; padding: 10px; border-radius: 5px; background-color: #f8f9fa; display: flex; align-items: center;">
            <div style="margin-right: 15px; font-size: 24px;">📄</div>
            <div>
                <div style="font-weight: bold;">{file_name}</div>
                <div style="color: #6c757d; font-size: 0.9em;">{description} • {size_str}</div>
            </div>
            <div style="margin-left: auto;">
                <a href="{file_path}" target="_blank" style="text-decoration: none; background-color: #007bff; color: white; padding: 5px 10px; border-radius: 3px;">View</a>
            </div>
        </div>
        """))
    else:
        log(f"File not found: {file_path}", "WARNING")

# Show a card for Google Drive links
def show_drive_link(url, title, description):
    display(HTML(f"""
    <div style="margin: 10px 0; padding: 15px; border-radius: 5px; background-color: #f0f7fb; border-left: 5px solid #3498db;">
        <div style="display: flex; align-items: center;">
            <div style="margin-right: 15px; font-size: 24px;">🔗</div>
            <div>
                <div style="font-weight: bold;">{title}</div>
                <div style="color: #6c757d; font-size: 0.9em;">{description}</div>
            </div>
            <div style="margin-left: auto;">
                <a href="{url}" target="_blank" style="text-decoration: none; background-color: #4285F4; color: white; padding: 5px 10px; border-radius: 3px;">Open in Google Drive</a>
            </div>
        </div>
    </div>
    """))

# Progress bar for long-running operations
def create_progress_bar(description="Processing"):
    bar = widgets.IntProgress(
        value=0,
        min=0,
        max=100,
        description=description,
        bar_style='info',
        style={'description_width': 'initial'}
    )
    display(bar)
    return bar

# Main execution function
def run_resume_generation():
    # Show welcome message
    display(HTML("""<h2 style="color: #3498db;">🚀 Resume Generation Process</h2>"""))
    log("Starting resume generation process...", "INFO")
    
    # === Step 1: Download from Google Drive ===
    show_section("Step 1: Retrieve Keyword Data", "Downloading data from Google Drive or using local file")
    
    if config["use_google_drive"] and google_drive_available:
        log(f"Downloading Google Sheet {config['sheet_id']} to {input_keywords_file}", "INFO")
        progress = create_progress_bar("Downloading")
        
        # Progress simulation for download (actual download doesn't report progress)
        for i in range(10):
            progress.value = i * 10
            time.sleep(0.1)
        
        success = google_drive.download_spreadsheet(config["sheet_id"], input_keywords_file)
        progress.value = 100
        
        if success:
            log("Google Sheet downloaded successfully", "SUCCESS")
            # Show a preview of the downloaded data
            try:
                keywords_df = pd.read_csv(input_keywords_file)
                display(HTML("<p><strong>Preview of downloaded keywords:</strong></p>"))
                display(keywords_df.head())
            except Exception as e:
                log(f"Error previewing downloaded file: {e}", "WARNING")
        else:
            log("Failed to download Google Sheet. Using existing file if available.", "WARNING")
    else:
        if not config["use_google_drive"]:
            log("Google Drive integration disabled in configuration. Using local file.", "INFO")
        elif not google_drive_available:
            log("Google Drive integration not available. Using local file.", "WARNING")
        
        # Check if local file exists
        if os.path.exists(input_keywords_file):
            log(f"Using existing keywords file: {input_keywords_file}", "INFO")
            try:
                keywords_df = pd.read_csv(input_keywords_file)
                display(HTML("<p><strong>Preview of local keywords file:</strong></p>"))
                display(keywords_df.head())
            except Exception as e:
                log(f"Error reading local file: {e}", "WARNING")
        else:
            log(f"Local keywords file not found: {input_keywords_file}", "ERROR")
            return
    
    # === Step 2: Process Keywords ===
    show_section("Step 2: Process Keywords", "Analyzing and extracting the most relevant keywords")
    log(f"Processing keywords from {input_keywords_file}", "INFO")
    
    try:
        progress = create_progress_bar("Processing keywords")
        progress.value = 30  # Initial progress
        
        # Process keywords
        high_df, low_df = keywords_processor.process_keywords(
            input_keywords_file, 
            processed_keywords_file
        )
        
        progress.value = 100
        log("Keywords processed successfully", "SUCCESS")
        
        # Display keyword results
        show_keywords_table(high_df, config["show_keyword_count"], "Top High-Priority Keywords")
        show_keywords_table(low_df, config["show_keyword_count"], "Top Low-Priority Keywords")
        
        show_file_link(processed_keywords_file, "Processed keywords data")
        
    except Exception as e:
        log(f"Error processing keywords: {e}", "ERROR")
        return
    
    # === Step 3: Apply Resume Structure ===
    show_section("Step 3: Apply Resume Structure", "Organizing resume sections according to configuration")
    log("Applying resume structure from configuration", "INFO")
    
    try:
        # Load the resume configuration
        try:
            with open(config_file, 'r') as f:
                resume_config = yaml.safe_load(f)
            
            # Show the section order
            section_order = resume_config.get('section_order', [])
            section_visibility = resume_config.get('section_visibility', {})
            
            # Display section information
            visible_sections = [s for s in section_order if section_visibility.get(s, True)]
            hidden_sections = [s for s in section_order if not section_visibility.get(s, True)]
            
            display(HTML(f"""
            <div style="padding: 10px; background-color: #f8f9fa; border-radius: 5px; margin: 10px 0;">
                <p><strong>Resume Structure Configuration:</strong></p>
                <p>Visible sections (in order): {', '.join(visible_sections)}</p>
                <p>Hidden sections: {', '.join(hidden_sections) if hidden_sections else 'None'}</p>
            </div>
            """))
            
        except Exception as e:
            log(f"Error loading resume configuration: {e}. Using default structure.", "WARNING")
            resume_config = None
        
        # Generate the resume structure
        progress = create_progress_bar("Applying structure")
        progress.value = 40
        
        # Generate the structured resume
        if resume_config:
            resume_generator.generate_resume(resume_config, output_resume_file)
            log("Resume structure applied successfully", "SUCCESS")
        else:
            log("Using default resume structure", "WARNING")
            # Just copy the template if no config is available
            with open(input_resume_file, 'r') as source, open(output_resume_file, 'w') as dest:
                dest.write(source.read())
        
        progress.value = 100
        
        # Show link to the structured resume
        show_file_link(output_resume_file, "Structured resume in markdown format")
        
    except Exception as e:
        log(f"Error applying resume structure: {e}", "ERROR")
        return
    
    # === Step 4: Update Resume with Keywords ===
    show_section("Step 4: Update Resume with Keywords", "Enhancing resume with processed keywords")
    log("Starting keyword update process", "INFO")
    
    try:
        # Load top keywords
        keywords_df = pd.read_csv(processed_keywords_file)
        high_priority = keywords_df[(keywords_df['priority'] == 'high')].sort_values('count', ascending=False).head(config["top_keywords"])
        keywords = high_priority['keyword'].tolist()
        
        log(f"Updating resume with top {len(keywords)} keywords", "INFO")
        
        # Update resume
        progress = create_progress_bar("Updating with keywords")
        progress.value = 50  # Initial progress
        
        update_resume.highlight_keywords(
            output_resume_file,  # Now we use the already structured resume
            output_resume_file,  # Overwrite with keyword-enhanced version
            keywords
        )
        
        progress.value = 100
        log("Resume updated with keywords successfully", "SUCCESS")
        
        # Show link to the updated markdown file
        show_file_link(output_resume_file, "Updated resume in markdown format")
        
    except Exception as e:
        log(f"Error updating resume with keywords: {e}", "ERROR")
        return
    
    # === Step 5: Generate PDF ===
    show_section("Step 5: Generate PDF", "Creating a professional PDF from the updated resume")
    
    if config["generate_pdf"]:
        log("Starting PDF generation", "INFO")
        
        try:
            progress = create_progress_bar("Generating PDF")
            progress.value = 20  # Initial progress
            
            # Generate PDF
            pdf_path = update_resume.generate_pdf(
                output_resume_file,
                os.path.dirname(output_pdf_file)
            )
            
            progress.value = 100
            
            if pdf_path and os.path.exists(pdf_path):
                log("PDF generation complete", "SUCCESS")
                show_file_link(pdf_path, "Generated resume PDF")
            else:
                # Check if PDF exists from previous runs
                if os.path.exists(output_pdf_file):
                    log("Using existing PDF file", "WARNING")
                    pdf_path = output_pdf_file
                    show_file_link(pdf_path, "Existing resume PDF")
                else:
                    log("PDF generation failed and no existing PDF found", "ERROR")
                    log("Consider using VS Code with Markdown PDF extension", "INFO")
                    pdf_path = None
        except Exception as e:
            log(f"Error during PDF generation: {e}", "ERROR")
            pdf_path = None
            
            # Check if PDF exists despite the error
            if os.path.exists(output_pdf_file):
                log(f"Using existing PDF despite error", "WARNING")
                pdf_path = output_pdf_file
                show_file_link(pdf_path, "Existing resume PDF")
    else:
        log("PDF generation skipped based on configuration", "INFO")
        pdf_path = None
    
    # === Step 6: Upload to Google Drive ===
    web_link = None
    if config["use_google_drive"] and google_drive_available and pdf_path and os.path.exists(pdf_path):
        show_section("Step 6: Upload to Google Drive", "Sharing your resume on Google Drive")
        log(f"Uploading PDF {pdf_path} to Google Drive folder {config['folder_id']}", "INFO")
        
        try:
            progress = create_progress_bar("Uploading to Google Drive")
            
            # Progress simulation for upload (actual upload doesn't report progress)
            for i in range(10):
                progress.value = i * 10
                time.sleep(0.1)
            
            web_link = google_drive.upload_pdf_to_drive(
                pdf_path, 
                config["folder_id"]
            )
            
            progress.value = 100
            
            if web_link:
                log("PDF uploaded successfully to Google Drive", "SUCCESS")
                show_drive_link(web_link, "Resume PDF on Google Drive", "Your resume has been shared on Google Drive")
            else:
                log("Failed to upload PDF to Google Drive", "WARNING")
        except Exception as e:
            log(f"Error uploading to Google Drive: {e}", "ERROR")
    elif config["use_google_drive"] and google_drive_available:
        if not pdf_path or not os.path.exists(pdf_path):
            log("Skipping Google Drive upload because PDF was not generated", "WARNING")
    else:
        if not config["use_google_drive"]:
            log("Google Drive upload skipped based on configuration", "INFO")
        elif not google_drive_available:
            log("Google Drive integration not available for upload", "WARNING")
    
    # === Final Summary ===
    show_section("✨ Process Complete", "Your resume has been generated successfully")
    
    summary_items = [
        f"<strong>Input keywords:</strong> {input_keywords_file}",
        f"<strong>Processed keywords:</strong> {processed_keywords_file}",
        f"<strong>Updated resume:</strong> {output_resume_file}"
    ]
    
    if pdf_path and os.path.exists(pdf_path):
        summary_items.append(f"<strong>Generated PDF:</strong> {pdf_path}")
    
    if web_link:
        summary_items.append(f"<strong>Google Drive link:</strong> <a href='{web_link}' target='_blank'>{web_link}</a>")
    
    show_summary_box("Resume Generation Summary", summary_items)
    
    # Show top 10 keywords used
    display(HTML("<h4 style='margin-top: 20px;'>Top 10 Keywords Used</h4>"))
    for i, keyword in enumerate(keywords[:10]):
        count = high_priority.iloc[i]['count']
        display(HTML(f"<div style='margin-left: 20px;'>{i+1}. <strong>{keyword}</strong> (Count: {count})</div>"))
    
    log("All processing completed successfully!", "SUCCESS")

# Run the resume generation process
run_resume_generation()

## 📊 Advanced Options

The cell below allows you to use the individual components separately for more control over the process.

In [ ]:
# This cell is optional - only run if you want to perform individual operations

import ipywidgets as widgets
from IPython.display import display, HTML
import yaml

def on_operation_selected(change):
    global operation_output
    operation_output.clear_output()
    
    with operation_output:
        if change.new == "download":
            sheet_id = widgets.Text(
                value=config["sheet_id"],
                description="Sheet ID:",
                style={'description_width': 'initial'}
            )
            output_path = widgets.Text(
                value=input_keywords_file,
                description="Output file:",
                style={'description_width': 'initial'}
            )
            download_button = widgets.Button(description="Download")
            display(HTML("<h3>Download from Google Sheet</h3>"))
            display(HTML("<p>Enter the Google Sheet ID and output file path:</p>"))
            display(sheet_id)
            display(output_path)
            display(download_button)
            
            def on_download_clicked(b):
                download_output.clear_output()
                with download_output:
                    try:
                        log(f"Downloading from Sheet ID: {sheet_id.value}", "INFO")
                        success = google_drive.download_spreadsheet(sheet_id.value, output_path.value)
                        if success:
                            log("Download successful!", "SUCCESS")
                            show_file_link(output_path.value, "Downloaded keywords file")
                        else:
                            log("Download failed.", "ERROR")
                    except Exception as e:
                        log(f"Error during download: {e}", "ERROR")
            
            download_button.on_click(on_download_clicked)
            download_output = widgets.Output()
            display(download_output)
            
        elif change.new == "process":
            input_path = widgets.Text(
                value=input_keywords_file,
                description="Input file:",
                style={'description_width': 'initial'}
            )
            output_path = widgets.Text(
                value=processed_keywords_file,
                description="Output file:",
                style={'description_width': 'initial'}
            )
            process_button = widgets.Button(description="Process Keywords")
            display(HTML("<h3>Process Keywords</h3>"))
            display(HTML("<p>Enter the input and output file paths:</p>"))
            display(input_path)
            display(output_path)
            display(process_button)
            
            def on_process_clicked(b):
                process_output.clear_output()
                with process_output:
                    try:
                        log(f"Processing keywords from {input_path.value}", "INFO")
                        high_df, low_df = keywords_processor.process_keywords(input_path.value, output_path.value)
                        log("Keywords processed successfully", "SUCCESS")
                        show_keywords_table(high_df, 10, "Top High-Priority Keywords")
                        show_keywords_table(low_df, 10, "Top Low-Priority Keywords")
                        show_file_link(output_path.value, "Processed keywords file")
                    except Exception as e:
                        log(f"Error processing keywords: {e}", "ERROR")
            
            process_button.on_click(on_process_clicked)
            process_output = widgets.Output()
            display(process_output)
            
        elif change.new == "structure":
            config_path = os.path.join(base_dir, "templates/resume_config.yaml")
            output_path = widgets.Text(
                value=os.path.join(base_dir, "data/output/temp_resume.md"),
                description="Preview file:",
                style={'description_width': 'initial'}
            )
            
            display(HTML("<h3>Manage Resume Structure</h3>"))
            display(HTML("<p>Load the current structure from the configuration file and make adjustments:</p>"))
            
            try:
                # Load current configuration
                with open(config_path, 'r') as f:
                    resume_config = yaml.safe_load(f)
                
                # Create section selection multi-select
                available_sections = resume_config.get('section_order', [])
                
                # Create a list of dictionaries for section ordering
                section_items = []
                for section in available_sections:
                    title = resume_config.get('section_titles', {}).get(section, section.capitalize())
                    visible = resume_config.get('section_visibility', {}).get(section, True)
                    section_items.append({
                        'name': section,
                        'title': title,
                        'visible': visible
                    })
                
                # Create widgets for each section
                section_widgets = []
                
                for item in section_items:
                    # Create section card
                    card = widgets.VBox([
                        widgets.HBox([
                            widgets.Label(value=f"{item['title']} ({item['name']})"),
                            widgets.Checkbox(
                                value=item['visible'],
                                description='Visible',
                                indent=False
                            )
                        ]),
                        widgets.HBox([
                            widgets.Button(description='↑ Move Up', layout=widgets.Layout(width='120px')),
                            widgets.Button(description='↓ Move Down', layout=widgets.Layout(width='120px'))
                        ])
                    ], layout=widgets.Layout(
                        border='1px solid #ddd',
                        padding='10px',
                        margin='5px 0',
                        border_radius='5px'
                    ))
                    
                    section_widgets.append(card)
                
                # Display section widgets
                sections_box = widgets.VBox(section_widgets)
                display(sections_box)
                
                # Preview and Save buttons
                preview_button = widgets.Button(
                    description='Preview Structure',
                    button_style='info',
                    layout=widgets.Layout(width='150px')
                )
                
                save_button = widgets.Button(
                    description='Save Structure',
                    button_style='success',
                    layout=widgets.Layout(width='150px')
                )
                
                display(widgets.HBox([preview_button, save_button]))
                
                # Output areas
                preview_output = widgets.Output()
                display(preview_output)
                
                # Button callbacks
                def on_preview_clicked(b):
                    preview_output.clear_output()
                    with preview_output:
                        try:
                            log("Generating preview with current structure...", "INFO")
                            
                            # Update section ordering based on current widgets
                            # [Implementation would go here]
                            
                            # Show preview
                            log("Preview generated", "SUCCESS")
                            
                        except Exception as e:
                            log(f"Error generating preview: {e}", "ERROR")
                
                def on_save_clicked(b):
                    preview_output.clear_output()
                    with preview_output:
                        try:
                            log("Saving structure configuration...", "INFO")
                            
                            # [Implementation would go here]
                            
                            log("Structure saved successfully", "SUCCESS")
                            log("Please go to the Resume Structure section for a better interface", "INFO")
                            
                        except Exception as e:
                            log(f"Error saving structure: {e}", "ERROR")
                
                preview_button.on_click(on_preview_clicked)
                save_button.on_click(on_save_clicked)
                
                # Add note about better interface
                display(HTML("""
                <div style="margin-top: 15px; padding: 15px; background-color: #fff3cd; border-radius: 5px; border-left: 5px solid #ffc107;">
                    <p><strong>Note:</strong> For a more comprehensive structure management interface, please use the <a href="#resume-structure">Resume Structure</a> section which provides better drag-and-drop capabilities and visual feedback.</p>
                </div>
                """))
                
            except Exception as e:
                log(f"Error loading resume configuration: {e}", "ERROR")
            
        elif change.new == "update":
            resume_path = widgets.Text(
                value=input_resume_file,
                description="Resume template:",
                style={'description_width': 'initial'}
            )
            keywords_path = widgets.Text(
                value=processed_keywords_file,
                description="Keywords file:",
                style={'description_width': 'initial'}
            )
            output_path = widgets.Text(
                value=output_resume_file,
                description="Output file:",
                style={'description_width': 'initial'}
            )
            update_button = widgets.Button(description="Update Resume")
            display(HTML("<h3>Update Resume</h3>"))
            display(HTML("<p>Enter the template, keywords, and output file paths:</p>"))
            display(resume_path)
            display(keywords_path)
            display(output_path)
            display(update_button)
            
            def on_update_clicked(b):
                update_output.clear_output()
                with update_output:
                    try:
                        log("Loading keywords...", "INFO")
                        keywords_df = pd.read_csv(keywords_path.value)
                        high_priority = keywords_df[(keywords_df['priority'] == 'high')].sort_values('count', ascending=False).head(config["top_keywords"])
                        keywords = high_priority['keyword'].tolist()
                        
                        log(f"Updating resume with top {len(keywords)} keywords", "INFO")
                        update_resume.highlight_keywords(resume_path.value, output_path.value, keywords)
                        log("Resume updated successfully", "SUCCESS")
                        show_file_link(output_path.value, "Updated resume file")
                    except Exception as e:
                        log(f"Error updating resume: {e}", "ERROR")
            
            update_button.on_click(on_update_clicked)
            update_output = widgets.Output()
            display(update_output)
            
        elif change.new == "pdf":
            input_path = widgets.Text(
                value=output_resume_file,
                description="Markdown file:",
                style={'description_width': 'initial'}
            )
            output_dir = widgets.Text(
                value=os.path.dirname(output_pdf_file),
                description="Output directory:",
                style={'description_width': 'initial'}
            )
            pdf_button = widgets.Button(description="Generate PDF")
            display(HTML("<h3>Generate PDF</h3>"))
            display(HTML("<p>Enter the markdown file and output directory:</p>"))
            display(input_path)
            display(output_dir)
            display(pdf_button)
            
            def on_pdf_clicked(b):
                pdf_output.clear_output()
                with pdf_output:
                    try:
                        log(f"Generating PDF from {input_path.value}", "INFO")
                        pdf_path = update_resume.generate_pdf(input_path.value, output_dir.value)
                        if pdf_path and os.path.exists(pdf_path):
                            log("PDF generated successfully", "SUCCESS")
                            show_file_link(pdf_path, "Generated PDF file")
                        else:
                            log("PDF generation failed", "ERROR")
                    except Exception as e:
                        log(f"Error generating PDF: {e}", "ERROR")
            
            pdf_button.on_click(on_pdf_clicked)
            pdf_output = widgets.Output()
            display(pdf_output)
            
        elif change.new == "upload":
            pdf_path = widgets.Text(
                value=output_pdf_file,
                description="PDF file:",
                style={'description_width': 'initial'}
            )
            folder_id = widgets.Text(
                value=config["folder_id"],
                description="Folder ID:",
                style={'description_width': 'initial'}
            )
            upload_button = widgets.Button(description="Upload to Drive")
            display(HTML("<h3>Upload to Google Drive</h3>"))
            display(HTML("<p>Enter the PDF file and Google Drive folder ID:</p>"))
            display(pdf_path)
            display(folder_id)
            display(upload_button)
            
            def on_upload_clicked(b):
                upload_output.clear_output()
                with upload_output:
                    try:
                        log(f"Uploading {pdf_path.value} to folder {folder_id.value}", "INFO")
                        web_link = google_drive.upload_pdf_to_drive(pdf_path.value, folder_id.value)
                        if web_link:
                            log("Upload successful!", "SUCCESS")
                            show_drive_link(web_link, "Resume on Google Drive", "Your resume has been uploaded to Google Drive")
                        else:
                            log("Upload failed", "ERROR")
                    except Exception as e:
                        log(f"Error uploading to Google Drive: {e}", "ERROR")
            
            upload_button.on_click(on_upload_clicked)
            upload_output = widgets.Output()
            display(upload_output)

# Create operation selection dropdown
operation = widgets.Dropdown(
    options=[
        ('Select an operation', 'none'),
        ('Download from Google Sheet', 'download'),
        ('Process Keywords', 'process'),
        ('Manage Resume Structure', 'structure'),
        ('Update Resume', 'update'),
        ('Generate PDF', 'pdf'),
        ('Upload to Google Drive', 'upload')
    ],
    value='none',
    description='Operation:',
    style={'description_width': 'initial'}
)

operation.observe(on_operation_selected, names='value')
display(HTML("<h3>Advanced Operations</h3>"))
display(HTML("<p>Select an individual operation to perform:</p>"))
display(operation)
operation_output = widgets.Output()
display(operation_output)

<a id="resume-structure"></a>
## 🧩 Resume Structure

Customize the structure and ordering of your resume sections below. Drag and drop sections to reorder them, or toggle visibility to show/hide sections.

In [ ]:
import yaml
import os
import ipywidgets as widgets
from IPython.display import display, HTML
import importlib

# Import resume_generator module - force reload to reflect any changes
if 'scripts.resume_generator' in sys.modules:
    importlib.reload(sys.modules['scripts.resume_generator'])
import scripts.resume_generator as resume_generator

# Define paths
config_path = os.path.join(base_dir, 'templates/resume_config.yaml')
temp_resume_path = os.path.join(base_dir, 'data/output/temp_resume.md')
resume_output_dir = os.path.join(base_dir, 'data/output')

# Load current configuration
def load_yaml_config(config_path):
    try:
        with open(config_path, 'r') as f:
            return yaml.safe_load(f)
    except Exception as e:
        log(f"Error loading configuration: {e}", "ERROR")
        return None

# Save configuration to YAML file
def save_yaml_config(config, config_path):
    try:
        with open(config_path, 'w') as f:
            yaml.dump(config, f, default_flow_style=False, sort_keys=False)
        return True
    except Exception as e:
        log(f"Error saving configuration: {e}", "ERROR")
        return False

# Load the configuration
resume_config = load_yaml_config(config_path)

if not resume_config:
    log("Could not load resume configuration. Please check the config file.", "ERROR")
else:
    # Create section widgets
    section_widgets = {}
    section_visibility = {}
    
    # Create a visual section card
    def create_section_card(section, title, is_visible=True):
        # Main container
        card = widgets.VBox([
            # Header with title and visibility toggle
            widgets.HBox([
                widgets.Label(value=title, style={'font_weight': 'bold'}),
                widgets.Checkbox(
                    value=is_visible,
                    description='Visible',
                    indent=False
                )
            ], layout=widgets.Layout(
                justify_content='space-between',
                width='100%'
            )),
            # Move up/down buttons
            widgets.HBox([
                widgets.Button(
                    description='↑ Move Up',
                    button_style='',
                    layout=widgets.Layout(width='48%')
                ),
                widgets.Button(
                    description='↓ Move Down',
                    button_style='',
                    layout=widgets.Layout(width='48%')
                )
            ], layout=widgets.Layout(
                justify_content='space-between',
                width='100%'
            ))
        ], layout=widgets.Layout(
            border='1px solid #ddd',
            padding='10px',
            margin='5px 0',
            border_radius='5px',
            width='100%'
        ))
        
        return card
    
    # Create header
    display(HTML("""
    <div style="background-color: #f5f5f5; padding: 15px; border-radius: 5px; margin-bottom: 20px; border-left: 5px solid #5cb85c;">
        <h3 style="margin-top: 0; color: #333;">Resume Section Order</h3>
        <p>Use the controls below to customize the structure of your resume. Changes will be saved to your resume configuration file.</p>
    </div>
    """))
    
    # Create section widgets
    section_container = widgets.VBox([])
    sections_list = []
    
    # Populate with current configuration
    for section in resume_config['section_order']:
        title = resume_config.get('section_titles', {}).get(section, section.capitalize())
        is_visible = resume_config.get('section_visibility', {}).get(section, True)
        
        section_card = create_section_card(section, title, is_visible)
        sections_list.append({'id': section, 'widget': section_card, 'title': title, 'visible': is_visible})
    
    # Update the section container
    def update_section_container():
        new_children = [item['widget'] for item in sections_list]
        section_container.children = new_children
    
    # Initialize the container
    update_section_container()
    
    # Set up button callbacks
    def setup_button_callbacks():
        for i, item in enumerate(sections_list):
            # Get both buttons from the card
            up_button = item['widget'].children[1].children[0]
            down_button = item['widget'].children[1].children[1]
            visibility_checkbox = item['widget'].children[0].children[1]
            
            # Store the section index for the callbacks
            up_button.section_index = i
            down_button.section_index = i
            visibility_checkbox.section_index = i
            
            # Clear existing handlers to avoid duplicates
            up_button._click_handlers.callbacks = []
            down_button._click_handlers.callbacks = []
            visibility_checkbox._click_handlers.callbacks = []
            
            # Define the callbacks using factory functions to create proper closures
            def make_up_click_handler(idx):
                def on_up_click(b):
                    if idx > 0:
                        # Swap with the previous section
                        sections_list[idx], sections_list[idx-1] = sections_list[idx-1], sections_list[idx]
                        # Update indexes for buttons
                        setup_button_callbacks()
                        # Update the UI
                        update_section_container()
                return on_up_click
            
            def make_down_click_handler(idx):
                def on_down_click(b):
                    if idx < len(sections_list) - 1:
                        # Swap with the next section
                        sections_list[idx], sections_list[idx+1] = sections_list[idx+1], sections_list[idx]
                        # Update indexes for buttons
                        setup_button_callbacks()
                        # Update the UI
                        update_section_container()
                return on_down_click
            
            def make_visibility_change_handler(idx):
                def on_visibility_change(change):
                    sections_list[idx]['visible'] = change['new']
                return on_visibility_change
            
            # Attach the callbacks using the factory functions
            up_button.on_click(make_up_click_handler(i))
            down_button.on_click(make_down_click_handler(i))
            visibility_checkbox.observe(make_visibility_change_handler(i), names='value')
    
    # Set up the initial callbacks
    setup_button_callbacks()
    
    # Save button
    save_button = widgets.Button(
        description='Save Structure',
        button_style='success',
        icon='check'
    )
    
    # Preview button
    preview_button = widgets.Button(
        description='Preview Structure',
        button_style='info',
        icon='eye'
    )
    
    # Status output area
    status_output = widgets.Output()
    
    # Preview output area
    preview_output = widgets.Output()
    
    # Save callback
    def on_save_click(b):
        status_output.clear_output()
        with status_output:
            # Update the config with the new order and visibility
            new_order = [item['id'] for item in sections_list]
            new_visibility = {item['id']: item['visible'] for item in sections_list}
            
            resume_config['section_order'] = new_order
            resume_config['section_visibility'] = new_visibility
            
            # Save back to the file
            if save_yaml_config(resume_config, config_path):
                log("Resume structure saved successfully!", "SUCCESS")
                display(HTML(f"""
                <div style="margin-top: 10px; padding: 10px; background-color: #d4edda; border-radius: 5px; color: #155724;">
                    <p><strong>Structure saved!</strong> Your resume sections will now appear in the following order:</p>
                    <ol>
                        {"".join(f"<li>{item['title']} {'(visible)' if item['visible'] else '(hidden)'}</li>" for item in sections_list)}
                    </ol>
                    <p>Run the resume generation process to apply these changes.</p>
                </div>
                """))
            else:
                log("Failed to save resume structure", "ERROR")
    
    # Preview callback
    def on_preview_click(b):
        preview_output.clear_output()
        with preview_output:
            # Update the config with the new order and visibility
            new_order = [item['id'] for item in sections_list]
            new_visibility = {item['id']: item['visible'] for item in sections_list}
            
            # Create a temporary config copy
            temp_config = resume_config.copy()
            temp_config['section_order'] = new_order
            temp_config['section_visibility'] = new_visibility
            
            # Generate a temporary preview
            try:
                # Create directory if needed
                os.makedirs(os.path.dirname(temp_resume_path), exist_ok=True)
                
                # Generate resume structure
                resume_generator.generate_resume(temp_config, temp_resume_path)
                
                # Read first part of the generated file
                with open(temp_resume_path, 'r') as f:
                    content = f.read(1000)  # Read first 1000 chars as a preview
                
                display(HTML(f"""
                <div style="margin-top: 10px;">
                    <h4>Resume Structure Preview:</h4>
                    <div style="margin-left: 20px; margin-top: 5px;">
                        <p><strong>Sections in order:</strong></p>
                        <ol>
                            {"".join(f"<li>{item['title']} {'✓' if item['visible'] else '✗'}</li>" for item in sections_list)}
                        </ol>
                    </div>
                </div>
                """))
                
                # Show preview file link
                show_file_link(temp_resume_path, "Preview of resume structure (click to view)")
                
            except Exception as e:
                log(f"Error generating preview: {e}", "ERROR")
    
    # Attach callbacks
    save_button.on_click(on_save_click)
    preview_button.on_click(on_preview_click)
    
    # Display the UI
    display(section_container)
    
    display(widgets.HBox([save_button, preview_button]))
    display(status_output)
    display(preview_output)
    
    # Add section explaining how to use this feature
    display(HTML("""
    <div style="margin-top: 20px; padding: 15px; background-color: #f9f9f9; border-radius: 5px;">
        <h4>How Resume Section Ordering Works</h4>
        <p>Use this feature to customize which sections appear in your resume and in what order. This is particularly useful for tailoring your resume to different job applications:</p>
        <ul>
            <li><strong>Technical roles:</strong> Put skills and projects first</li>
            <li><strong>Leadership positions:</strong> Emphasize experience and certifications</li>
            <li><strong>Academic applications:</strong> Feature education and publications prominently</li>
        </ul>
        <p>After saving your preferred structure, run the resume generation process to apply these changes to your resume.</p>
    </div>
    """))

## 📋 Project Information

<div style="background-color: #f0f0f0; padding: 15px; border-radius: 5px;">
    <h3 style="margin-top: 0;">About the Resume Generator</h3>
    <p>This tool helps you maintain a keyword-optimized resume by analyzing job descriptions, extracting relevant keywords, and updating your resume to highlight your skills that match these keywords. The process is fully automated from downloading data to uploading the final PDF.</p>
    
    <h4>Key Features</h4>
    <ul>
        <li><strong>Google Drive Integration:</strong> Download keyword data directly from Google Sheets and upload the generated PDF back to Google Drive</li>
        <li><strong>Smart Keyword Processing:</strong> Preserves multi-word terms like "machine learning" and prioritizes keywords based on frequency</li>
        <li><strong>PDF Generation:</strong> Creates a professional PDF resume using pandoc with proper formatting</li>
        <li><strong>Interactive Interface:</strong> Run the entire process with a single click, with visual feedback and progress indicators</li>
        <li><strong>Customizable Structure:</strong> Rearrange your resume sections for maximum impact based on the job requirements</li>
    </ul>
</div>

<div style="margin-top: 20px; text-align: center; color: #777; font-size: 0.9em;">
    <p>Created by Malachi Dunn • © 2024 • <a href="https://github.com/malachi-mindwyre/resume">GitHub Repository</a></p>
</div>